# 🏥 Hospital Equipment Demand Prediction

Trains a model to predict `value` — how many times a given equipment will be locked
in the **next 3 days** (`date → recorded_at`).

### Schema of `equipment_usage_snapshot`
| Column | Description |
|---|---|
| `equipment` | Equipment name (e.g. Ventilators) |
| `date` | Reference date = recorded_at − 3 days |
| `recorded_at` | Timestamp when row was saved |
| `total_patients_last_day` | Admissions in [date−1 → date] |
| `total_patients_last_7_days` | Admissions in [date−7 → date] |
| `total_usage_last_day` | This equipment locked in [date−1 → date] |
| `total_usage_last_7_days` | This equipment locked in [date−7 → date] |
| `weather` | e.g. "Sunny, 36°C" |
| `is_holiday` | Any Indian public holiday in [date → recorded_at] |
| `is_weekend` | Is `date` a Saturday or Sunday |
| `value` | **TARGET**: equipment locked in [date → recorded_at] (3-day forward window) |

## 1. Install Dependencies

In [ ]:
!pip install pandas numpy scikit-learn psycopg2-binary sqlalchemy matplotlib seaborn joblib

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')
print('✅ Libraries imported.')

## 3. Connect to PostgreSQL & Load Data

In [ ]:
DB_USER     = 'postgres'
DB_PASSWORD = 'your_password'   # ← change
DB_HOST     = 'localhost'
DB_PORT     = '5432'
DB_NAME     = 'hospital_db'     # ← change

engine = create_engine(f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
df = pd.read_sql('SELECT * FROM equipment_usage_snapshot ORDER BY recorded_at', engine)

print(f'✅ Loaded {len(df):,} rows')
df.head()

## 4. Exploratory Analysis

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
print('\nValue distribution:')
print(df['value'].describe())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['equipment'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', title='Rows per equipment')
df['value'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='coral', title='Target (value) distribution')
df['is_weekend'].value_counts().plot(kind='bar', ax=axes[2], color='teal', title='Weekend distribution')

for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df_model = df.copy()

# Parse date features
df_model['date'] = pd.to_datetime(df_model['date'])
df_model['day_of_week'] = df_model['date'].dt.dayofweek   # 0=Mon…6=Sun
df_model['month']       = df_model['date'].dt.month
df_model['day']         = df_model['date'].dt.day

# Parse weather: "Sunny, 36°C" → condition + temperature
def parse_condition(w):
    if pd.isna(w) or w == 'Unavailable': return 'Unknown'
    return str(w).split(',')[0].strip()

def parse_temp(w):
    if pd.isna(w) or w == 'Unavailable': return np.nan
    match = re.search(r'([+-]?\d+)', str(w).split(',')[-1])
    return float(match.group(1)) if match else np.nan

df_model['weather_condition'] = df_model['weather'].apply(parse_condition)
df_model['temperature_c']     = df_model['weather'].apply(parse_temp)
df_model['temperature_c'].fillna(df_model['temperature_c'].median(), inplace=True)

# Encode categorical columns
le_equipment = LabelEncoder()
le_weather   = LabelEncoder()
df_model['equipment_enc']    = le_equipment.fit_transform(df_model['equipment'])
df_model['weather_enc']      = le_weather.fit_transform(df_model['weather_condition'])

# Boolean flags → int
df_model['is_holiday'] = df_model['is_holiday'].astype(int)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

print('✅ Feature engineering done.')
df_model[['equipment', 'date', 'day_of_week', 'month', 'weather_condition',
           'temperature_c', 'is_holiday', 'is_weekend']].head()

## 6. Prepare Features & Target

In [ ]:
FEATURES = [
    'equipment_enc',
    'day_of_week',
    'month',
    'day',
    'total_patients_last_day',
    'total_patients_last_7_days',
    'total_usage_last_day',
    'total_usage_last_7_days',
    'weather_enc',
    'temperature_c',
    'is_holiday',
    'is_weekend',
]

TARGET = 'value'

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 7. Train Models

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print('── Random Forest ──────────────────')
print(f'  MAE  : {mean_absolute_error(y_test, rf_pred):.4f}')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, rf_pred)):.4f}')
print(f'  R²   : {r2_score(y_test, rf_pred):.4f}')

In [ ]:
gb = GradientBoostingRegressor(n_estimators=200, max_depth=5, learning_rate=0.05, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)

print('── Gradient Boosting ──────────────')
print(f'  MAE  : {mean_absolute_error(y_test, gb_pred):.4f}')
print(f'  RMSE : {np.sqrt(mean_squared_error(y_test, gb_pred)):.4f}')
print(f'  R²   : {r2_score(y_test, gb_pred):.4f}')

## 8. Select Best Model & Feature Importance

In [ ]:
rf_r2 = r2_score(y_test, rf_pred)
gb_r2 = r2_score(y_test, gb_pred)
best_model = rf if rf_r2 >= gb_r2 else gb
best_name  = 'Random Forest' if rf_r2 >= gb_r2 else 'Gradient Boosting'
print(f'🏆 Best model: {best_name}  (R² = {max(rf_r2, gb_r2):.4f})')

importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values()
importances.plot(kind='barh', color='teal', figsize=(8, 5), title=f'Feature Importance — {best_name}')
plt.tight_layout(); plt.show()

## 9. Actual vs Predicted

In [ ]:
best_pred = best_model.predict(X_test)
plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_pred, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='Perfect')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted — {best_name}')
plt.legend(); plt.tight_layout(); plt.show()

## 10. Save Model

In [ ]:
joblib.dump(best_model,   'equipment_demand_model.pkl')
joblib.dump(le_equipment, 'label_encoder_equipment.pkl')
joblib.dump(le_weather,   'label_encoder_weather.pkl')
print('✅ Saved: equipment_demand_model.pkl')

## 11. Inference Example

In [ ]:
model    = joblib.load('equipment_demand_model.pkl')
le_eq    = joblib.load('label_encoder_equipment.pkl')
le_wx    = joblib.load('label_encoder_weather.pkl')

# ── Fill in your values ─────────────────────────────────────────────────
sample = {
    'equipment'                 : 'Ventilators',
    'day_of_week'               : 2,        # 0=Mon…6=Sun
    'month'                     : 4,
    'day'                       : 12,
    'total_patients_last_day'   : 3,
    'total_patients_last_7_days': 18,
    'total_usage_last_day'      : 1,
    'total_usage_last_7_days'   : 7,
    'weather_condition'         : 'Sunny',
    'temperature_c'             : 36,
    'is_holiday'                : 0,
    'is_weekend'                : 0,
}
# ────────────────────────────────────────────────────────────────────────

eq_enc = le_eq.transform([sample['equipment']])[0]
wx_enc = le_wx.transform([sample['weather_condition']])[0]

X_new = pd.DataFrame([{
    'equipment_enc'              : eq_enc,
    'day_of_week'                : sample['day_of_week'],
    'month'                      : sample['month'],
    'day'                        : sample['day'],
    'total_patients_last_day'    : sample['total_patients_last_day'],
    'total_patients_last_7_days' : sample['total_patients_last_7_days'],
    'total_usage_last_day'       : sample['total_usage_last_day'],
    'total_usage_last_7_days'    : sample['total_usage_last_7_days'],
    'weather_enc'                : wx_enc,
    'temperature_c'              : sample['temperature_c'],
    'is_holiday'                 : sample['is_holiday'],
    'is_weekend'                 : sample['is_weekend'],
}])

pred = model.predict(X_new)[0]
print(f"📊 Predicted demand for '{sample['equipment']}' over next 3 days: {pred:.2f} units")